In [1]:
import numpy as np
import pandas as pd
import pickle as pc
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB, CategoricalNB, ComplementNB
from sklearn.preprocessing import StandardScaler, MinMaxScaler, KBinsDiscretizer
from sklearn.metrics import confusion_matrix, classification_report, f1_score, roc_auc_score

In [2]:
Dataset = pd.read_csv("CKD.csv")
Dataset = pd.get_dummies(Dataset, dtype = int, drop_first = True )
Dataset['classification_yes'].value_counts()

classification_yes
1    249
0    150
Name: count, dtype: int64

In [3]:
Dataset.columns

Index(['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes', 'classification_yes'],
      dtype='str')

In [4]:
Independent = Dataset[['age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo', 'pcv',
       'wc', 'rc', 'sg_b', 'sg_c', 'sg_d', 'sg_e', 'rbc_normal', 'pc_normal',
       'pcc_present', 'ba_present', 'htn_yes', 'dm_yes', 'cad_yes',
       'appet_yes', 'pe_yes', 'ane_yes']]
Dependent = Dataset['classification_yes']
print("Independent shape :" ,Independent.shape)
print("Dependent shape :" ,Dependent.shape)

Independent shape : (399, 27)
Dependent shape : (399,)


In [5]:
X_Train, X_Test, Y_Train, Y_Test = train_test_split(Independent, Dependent, test_size = 0.30, random_state = 0)

In [6]:
# SCALERS
# Gaussian NB
scaler_standard = StandardScaler()
X_Train_G = scaler_standard.fit_transform(X_Train)
X_Test_G = scaler_standard.transform(X_Test)

# Multinomial, Complement, Bernoulli
# Need positive values

scaler_minmax = MinMaxScaler()
X_Train_M = scaler_minmax.fit_transform(X_Train)
X_Test_M = scaler_minmax.transform(X_Test)


In [7]:
# MODEL LIST
models = {

    "GaussianNB": (
        GaussianNB(),
        {
            'var_smoothing': [
                1e-09,
                1e-08,
                1e-07,
                1e-06,
                1e-05
            ]
        },
        X_Train_G,
        X_Test_G
    ),

    "MultinomialNB": (
        MultinomialNB(),
        {
            'alpha': [0.01, 0.1, 1, 10, 100],
            'fit_prior': [True, False]
        },
        X_Train_M,
        X_Test_M
    ),

    "BernoulliNB": (
        BernoulliNB(),
        {
            'alpha': [0.01, 0.1, 1, 10],
            'fit_prior': [True, False]
        },
        X_Train_M,
        X_Test_M
    ),

    "ComplementNB": (
        ComplementNB(),
        {
            'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
            'fit_prior': [True, False],
            'norm': [True, False]
        },
        X_Train_M,
        X_Test_M
    )
}


In [8]:
# MODEL COMPARISON
Results = []

best_model = None
best_scaler = None
best_f1 = 0

for model_name, (model, params, Xtrain, Xtest) in models.items():

    print("\n" + "="*60)
    print(model_name)
    print("="*60)

    Grid_Classifier = GridSearchCV(
        estimator=model,
        param_grid=params,
        scoring='f1_weighted',
        cv=5,
        n_jobs=-1,
        refit=True
    )

    Grid_Classifier.fit(Xtrain, Y_Train)

    grid_Y_Pred = Grid_Classifier.predict(Xtest)

    fl_macro_avg = f1_score(
        Y_Test,
        grid_Y_Pred,
        average='weighted'
    )

    auc = roc_auc_score(
        Y_Test,
        Grid_Classifier.predict_proba(Xtest)[:, 1]
    )

    print("Best Parameters :", Grid_Classifier.best_params_)
    print("F1 Score :", fl_macro_avg)
    print("ROC AUC :", auc)

    print("\nConfusion Matrix")
    print(confusion_matrix(Y_Test, grid_Y_Pred))

    print("\nClassification Report")
    print(classification_report(Y_Test, grid_Y_Pred))

    Results.append(
        [model_name,
         fl_macro_avg,
         auc,
         Grid_Classifier.best_params_]
    )

    if fl_macro_avg > best_f1:

        best_f1 = fl_macro_avg
        best_model = Grid_Classifier.best_estimator_

        if model_name == "GaussianNB":
            best_scaler = scaler_standard
        else:
            best_scaler = scaler_minmax




GaussianNB
Best Parameters : {'var_smoothing': 1e-07}
F1 Score : 0.9751481237656352
ROC AUC : 1.0

Confusion Matrix
[[45  0]
 [ 3 72]]

Classification Report
              precision    recall  f1-score   support

           0       0.94      1.00      0.97        45
           1       1.00      0.96      0.98        75

    accuracy                           0.97       120
   macro avg       0.97      0.98      0.97       120
weighted avg       0.98      0.97      0.98       120


MultinomialNB
Best Parameters : {'alpha': 0.01, 'fit_prior': True}
F1 Score : 0.9834018801410106
ROC AUC : 1.0

Confusion Matrix
[[45  0]
 [ 2 73]]

Classification Report
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        45
           1       1.00      0.97      0.99        75

    accuracy                           0.98       120
   macro avg       0.98      0.99      0.98       120
weighted avg       0.98      0.98      0.98       120


BernoulliNB
Be

In [9]:
# RESULT TABLE
Result_DF = pd.DataFrame(
    Results,
    columns=[
        'Model',
        'F1 Score',
        'ROC_AUC',
        'Best Parameters'
    ]
)

print("\n")
print(Result_DF.sort_values(
    by='F1 Score',
    ascending=False
))

print("\nBest Model Selected")
print(best_model)
print("Best F1 Score :", best_f1)



           Model  F1 Score   ROC_AUC  \
1  MultinomialNB  0.983402  1.000000   
3   ComplementNB  0.983402  1.000000   
0     GaussianNB  0.975148  1.000000   
2    BernoulliNB  0.975148  0.996741   

                                     Best Parameters  
1                 {'alpha': 0.01, 'fit_prior': True}  
3  {'alpha': 0.001, 'fit_prior': True, 'norm': Fa...  
0                           {'var_smoothing': 1e-07}  
2                 {'alpha': 0.01, 'fit_prior': True}  

Best Model Selected
MultinomialNB(alpha=0.01)
Best F1 Score : 0.9834018801410106


In [10]:
pc.dump(best_model,open("MLC_Assignment_Naive_bayes_Model_Grid.sav",'wb'))
pc.dump(best_scaler,open("MLC_Assignment_Naive_bayes_scaler.sav",'wb'))


In [11]:
Loaded_model = pc.load(open ("MLC_Assignment_Naive_bayes_Model_Grid.sav",'rb'))
Loaded_scaler = pc.load(open ("MLC_Assignment_Naive_bayes_scaler.sav",'rb'))


In [15]:
# Numerical Inputs
age = float(input("Age: "))
bp = float(input("BP: "))
al = float(input("Albumin (al): "))
su = float(input("Sugar (su): "))
bgr = float(input("Blood Glucose Random (bgr): "))
bu = float(input("Blood Urea (bu): "))
sc = float(input("Serum Creatinine (sc): "))
sod = float(input("Sodium (sod): "))
pot = float(input("Potassium (pot): "))
hrmo = float(input("Hemoglobin (hrmo): "))
pcv = float(input("Packed Cell Volume (pcv): "))
wc = float(input("White Blood Cell Count (wc): "))
rc = float(input("Red Blood Cell Count (rc): "))

# One-Hot Encoded Inputs
sg_b = int(input("sg_b (0/1): "))
sg_c = int(input("sg_c (0/1): "))
sg_d = int(input("sg_d (0/1): "))
sg_e = int(input("sg_e (0/1): "))

rbc_normal = int(input("rbc_normal (0/1): "))
pc_normal = int(input("pc_normal (0/1): "))
pcc_present = int(input("pcc_present (0/1): "))
ba_present = int(input("ba_present (0/1): "))

htn_yes = int(input("htn_yes (0/1): "))
dm_yes = int(input("dm_yes (0/1): "))
cad_yes = int(input("cad_yes (0/1): "))

appet_yes = int(input("appet_yes (0/1): "))
pe_yes = int(input("pe_yes (0/1): "))
ane_yes = int(input("ane_yes (0/1): "))

# Create DataFrame
new_data = pd.DataFrame(
    [[
        age, bp, al, su, bgr, bu, sc, sod, pot, hrmo,
        pcv, wc, rc,
        sg_b, sg_c, sg_d, sg_e,
        rbc_normal, pc_normal,
        pcc_present, ba_present,
        htn_yes, dm_yes, cad_yes,
        appet_yes, pe_yes, ane_yes
    ]],
    columns=[
        'age', 'bp', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hrmo',
        'pcv', 'wc', 'rc',
        'sg_b', 'sg_c', 'sg_d', 'sg_e',
        'rbc_normal', 'pc_normal',
        'pcc_present', 'ba_present',
        'htn_yes', 'dm_yes', 'cad_yes',
        'appet_yes', 'pe_yes', 'ane_yes'
    ]
)

# Scale data
Preinput = Loaded_scaler.transform(new_data)

# Predict
Future_Predict=Loaded_model.predict(Preinput) # change the paramter,play with it.
probability = Loaded_model.predict_proba(Preinput)
if Future_Predict[0] == 1:
    print("\nPrediction: YES, Patient is likely to have CKD")
    print(f"Confidence : {probability[0][1]*100:.2f}%")
else:
    print("\nPrediction: NO, Patient is unlikely to have CKD")
    print(f"Confidence : {probability[0][0]*100:.2f}%")

Age:  12
BP:  80
Albumin (al):  0
Sugar (su):  0
Blood Glucose Random (bgr):  100
Blood Urea (bu):  26
Serum Creatinine (sc):  .6
Sodium (sod):  137
Potassium (pot):  4.4
Hemoglobin (hrmo):  15.8
Packed Cell Volume (pcv):  49
White Blood Cell Count (wc):  6600
Red Blood Cell Count (rc):  5.4
sg_b (0/1):  0
sg_c (0/1):  0
sg_d (0/1):  0
sg_e (0/1):  0
rbc_normal (0/1):  1
pc_normal (0/1):  1
pcc_present (0/1):  0
ba_present (0/1):  0
htn_yes (0/1):  0
dm_yes (0/1):  0
cad_yes (0/1):  0
appet_yes (0/1):  1
pe_yes (0/1):  0
ane_yes (0/1):  0



Prediction: NO, Patient is unlikely to have CKD
Confidence : 92.78%
